# 📊 DriftGuard — Easy Loan Dataset & Model Guide

Welcome! This notebook provides an easy, step-by-step walkthrough to explore the **Loan Application Dataset**, understand raw features, examine the **3-way Train/Val/Test partitions**, and see how the **XGBoost AI Model** predicts loan default risk.

---

## Step 1: Import Python Libraries

We use `sys` & `pathlib` for environment setup, `pandas` to view tables, `joblib` to load trained models, and `sklearn` to compute accuracy.

In [ ]:
import sys
from pathlib import Path
# Ensure project root is in sys.path for importing 'src' module
sys.path.insert(0, str(Path('../').resolve()))

import pandas as pd
import joblib
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

print("✅ All required libraries imported successfully!")

---

## Step 2: Load and View Raw Customer Data

Let's load the raw loan application dataset from `dataset/raw/train_u6lujuX_CVtuZ9i.csv`.

In [ ]:
raw_csv_path = Path('../dataset/raw/train_u6lujuX_CVtuZ9i.csv')
raw_df = pd.read_csv(raw_csv_path)

print(f"📌 Total Applicants (Rows): {raw_df.shape[0]}")
print(f"📌 Total Columns:           {raw_df.shape[1]}")
print("\nFirst 5 Applicant Records:")
raw_df.head()

### Column Explanation Guide:
- `Loan_ID`: Unique ID for each applicant
- `Gender`: Male or Female
- `Married`: Applicant marital status (Yes / No)
- `Dependents`: Number of family dependents (0, 1, 2, 3+)
- `Education`: Graduate or Not Graduate
- `Self_Employed`: Self-employed status (Yes / No)
- `ApplicantIncome`: Main applicant monthly income
- `CoapplicantIncome`: Secondary co-applicant monthly income
- `LoanAmount`: Requested loan amount (in thousands)
- `Loan_Amount_Term`: Duration of repayment (in months, e.g. 360 = 30 years)
- `Credit_History`: Credit history score (1.0 = good history, 0.0 = bad history)
- `Property_Area`: Location type (Urban, Semiurban, Rural)
- `Loan_Status`: **Target Variable** (`Y` = Approved, `N` = Rejected / Default Risk)

---

## Step 3: Check Target Distribution (`Loan_Status`)

How many applicants were Approved (`Y`) vs Rejected (`N`)?

In [ ]:
status_counts = raw_df['Loan_Status'].value_counts()
status_percentages = raw_df['Loan_Status'].value_counts(normalize=True) * 100

summary_df = pd.DataFrame({
    'Applicant Count': status_counts,
    'Percentage (%)': status_percentages.round(2)
})
print("📊 Loan Status Summary:")
summary_df

---

## Step 4: Check Missing Data in Raw Features

Before training an ML model, missing values must be imputed.

In [ ]:
missing_counts = raw_df.isnull().sum()
missing_summary = missing_counts[missing_counts > 0].reset_index()
missing_summary.columns = ['Feature Name', 'Missing Rows Count']
print("⚠️ Features with Missing Data:")
missing_summary

---

## Step 5: Explore Preprocessed 3-Way Partitions (Train, Val, Test)

Our pipeline cleaned missing data, scaled numeric variables (`StandardScaler`), encoded text columns (`OneHotEncoder`), and created 3 stratified partitions:
1. `train.parquet` (70% — 429 records) for model fitting
2. `val.parquet` (15% — 92 records) for early stopping evaluation
3. `test.parquet` (15% — 93 records) for final unbiased accuracy evaluation

In [ ]:
train_df = pd.read_parquet('../dataset/processed/train.parquet')
val_df   = pd.read_parquet('../dataset/processed/val.parquet')
test_df  = pd.read_parquet('../dataset/processed/test.parquet')

print(f"🟢 Train Set Shape:      {train_df.shape} (70% of dataset)")
print(f"🟡 Validation Set Shape: {val_df.shape}   (15% of dataset)")
print(f"🔴 Test Set Shape:       {test_df.shape}   (15% of dataset)")

print("\nFirst 5 Processed Training Rows (Clean & Scaled Features):")
train_df.head()

---

## Step 6: Test Predictions with Trained XGBoost Model

Now let's load `models/model.joblib` and predict default risk probabilities on 5 test applicants.

In [ ]:
model = joblib.load('../models/model.joblib')

# Separate features X from ground truth y
X_test = test_df.drop(columns=['Loan_Status'])
y_test = test_df['Loan_Status'].astype(int)

# Predict probabilities and binary labels
probabilities = model.predict_proba(X_test)
predictions = model.predict(X_test)

# Display sample prediction table for first 5 test applicants
sample_results = pd.DataFrame({
    'Actual Status': y_test.head(5).map({1: 'Approved (1)', 0: 'Rejected (0)'}),
    'Predicted Status': pd.Series(predictions[:5]).map({1: 'Approved (1)', 0: 'Rejected (0)'}),
    'Approval Probability (%)': (probabilities[:5, 1] * 100).round(2),
    'Rejection Probability (%)': (probabilities[:5, 0] * 100).round(2)
})

print(f"🤖 Model Framework: {model.model_type.upper()}")
print(f"🎯 Test Accuracy:   {accuracy_score(y_test, predictions)*100:.2f}%")
print(f"📈 Test ROC-AUC:    {roc_auc_score(y_test, probabilities[:, 1]):.4f}")

print("\nSample Prediction Results:")
sample_results